# Семинар 1. Первый разговор с моделью

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sblenlkj/summer-agent-school-seminars-1-2/blob/main/notebooks/seminar_1_8_9.ipynb)

## 1. Что мы сделаем сегодня

1. вспомним, что такое языковая модель (LLM);
2. получим ключ и временный токен для доступа к модели;
3. отправим первый запрос и получим ответ;
4. будем менять **промпт** (текст запроса) и смотреть, как меняется ответ;
5. соберём маленький чат;
6. в конце увидим, как модель может работать внутри обычной программы.

Главное правило семинара:

> Сначала запускаем ячейку. Потом смотрим, что получилось. Потом меняем текст и запускаем снова.


## 2. Коротко: что такое языковая модель

**Языковая модель (LLM)** — это программа, которая умеет продолжать текст.

Она не «думает» как человек. Она собирает ответ **по маленьким кусочкам** — эти кусочки называют **токенами**. Токен — это примерно часть слова.

Например, слово «привет» модель может видеть как кусочки: `при` + `вет`.

Модель смотрит на наш текст и каждый раз предсказывает следующий кусочек-токен. Так, токен за токеном, и получается целый ответ.

Сегодня нам не нужно это запоминать. Нам важно **увидеть это вживую**: дальше мы даже посмотрим, сколько токенов потратила модель на один ответ.


## 3. Проверяем, что Colab работает

Перед тем как обращаться к модели, убедимся, что код вообще запускается.

Нажмите кнопку ▶ слева от ячейки или сочетание клавиш:

- `Ctrl` (или `Cmd`) + `Enter` — запустить ячейку;
- `Shift` + `Enter` — запустить ячейку и перейти к следующей.


In [ ]:
print("Colab работает!")
print("Сегодня мы будем отправлять запросы к языковой модели.")

## 4. Ключ и токен

Мы работаем с моделью **GigaChat** через интернет. Чтобы модель понимала, что запрос пришёл именно от нас, нужен **ключ**.

Ключ — это как **логин и пароль** вместе. По нему нам разрешают обращаться к модели (в рамках курса всё бесплатно).

У GigaChat есть особенность. Ключ из личного кабинета **не отправляется напрямую** в модель. Сначала мы меняем его на временный пропуск — **access token**:

```text
ключ  →  получаем access token  →  с этим токеном отправляем запрос  →  ответ модели
```

- **ключ** — как долгосрочный пропуск (живёт долго);
- **access token** — как временный билетик: действует около 30 минут, потом его нужно получить заново.

**Если позже появится ошибка про токен** — просто заново запустите ячейку, где мы получаем access token.

---

**TODO (преподаватель):** показать, где взять ключ авторизации GigaChat.

---

Запустите ячейку ниже и **вставьте ключ** в появившееся поле, потом нажмите `Enter`.


In [ ]:
# Адреса серверов GigaChat — это просто настройки, их менять не нужно.
AUTH_URL = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"
BASE_URL = "https://gigachat.devices.sberbank.ru/api/v1/chat/completions"
MODELS_URL = "https://gigachat.devices.sberbank.ru/api/v1/models"

# Название модели. Для семинара берём самую быструю.
MODEL_NAME = "GigaChat-2"

# Вставляем ключ авторизации. Это НЕ access token, а ключ, по которому мы его получим.
AUTH_KEY = input("Вставьте ключ авторизации GigaChat и нажмите Enter: ")

print("Настройки сохранены. Модель:", MODEL_NAME)

**Важно про ключ:** ключ нельзя выкладывать в открытый доступ и показывать чужим людям. Мы вставляем его через поле ввода, в тексте ноутбука он не сохраняется.


In [ ]:
from IPython.display import Image, display

try:
    display(Image(filename="../images/api_key_meme.png", height=400, width=500))
except Exception:
    print("Мем не нашёлся :(")

### Получаем access token

Эта ячейка — техническая. Здесь мы меняем ключ на временный токен. **Её не нужно понимать построчно — просто запустите.**

Если всё хорошо, увидите «Статус ответа: 200» и сообщение, что токен получен.


In [ ]:
import uuid
import requests
import warnings
from urllib3.exceptions import InsecureRequestWarning

# В Colab иногда ругается на сертификаты — для учебного семинара отключаем это предупреждение.
warnings.filterwarnings("ignore", category=InsecureRequestWarning)


def get_access_token(auth_key):
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Accept": "application/json",
        "RqUID": str(uuid.uuid4()),
        "Authorization": f"Basic {auth_key}",
    }
    data = {"scope": "GIGACHAT_API_PERS"}

    response = requests.post(AUTH_URL, headers=headers, data=data, verify=False)
    print("Статус ответа:", response.status_code)

    if response.status_code != 200:
        print("Не получилось получить токен. Проверьте ключ.")
        print(response.text)
        return None

    return response.json()["access_token"]


ACCESS_TOKEN = get_access_token(AUTH_KEY)

if ACCESS_TOKEN:
    print("Access token получен. Можно работать с моделью!")
else:
    print("Access token не получен. Проверьте ключ и запустите ячейку снова.")

Access token живёт около 30 минут. Если позже модель перестанет отвечать и появится ошибка про токен — просто запустите ячейку выше ещё раз.


## 5. Первый запрос к модели

Теперь отправим модели первый вопрос.

Запрос к модели — это небольшой «пакет данных»: в нём указано, **какая модель** и **какое сообщение** мы ей шлём. Запустите ячейку и посмотрите на ответ.


In [ ]:
# Что мы отправляем модели
payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "user", "content": "Привет! Объясни простыми словами, что такое API."}
    ],
    "temperature": 0.7,
    "max_tokens": 512,
}

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

response = requests.post(BASE_URL, headers=headers, json=payload, verify=False)

print("Статус ответа:", response.status_code)
print()
print(response.text)

### Что мы увидели?

Модель ответила, но ответ выглядит как большой непонятный текст со скобками. Это нормально — такой формат называется **JSON**. Это просто способ аккуратно упаковать данные.

Внутри этого ответа лежит не только текст модели, но и служебная информация. Например — **сколько токенов** (тех самых кусочков из начала семинара) потратила модель. Давайте достанем это.


In [ ]:
data = response.json()  # превращаем ответ в удобные данные

# Сколько токенов потратила модель на этот ответ:
print("Потрачено токенов:", data["usage"])
print()

# А вот так достаём САМ ТЕКСТ ответа модели:
answer = data["choices"][0]["message"]["content"]
print("Ответ модели:")
print(answer)

Видите строчку `data["choices"][0]["message"]["content"]`? Это «адрес», по которому внутри ответа лежит текст модели. На следующем семинаре мы разберём, **почему адрес такой длинный** и как это устроено.

Сейчас главное: мы научились доставать из ответа именно текст.


## 6. Удобная функция `ask_model`

Каждый раз писать весь этот код неудобно. Поэтому спрячем всю сложность в одну **функцию** `ask_model`.

Дальше мы будем просто писать `ask_model("наш вопрос")` и получать текст ответа. Запустите ячейку — она ничего не выводит, просто «готовит» функцию.


In [ ]:
def ask_model(prompt, temperature=0.7, max_tokens=512):
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

    response = requests.post(BASE_URL, headers=headers, json=payload, verify=False)

    if response.status_code != 200:
        print("Ошибка запроса. Статус:", response.status_code)
        if response.status_code == 401:
            print("Возможно, истёк токен. Перезапустите ячейку получения токена.")
        return None

    data = response.json()
    return data["choices"][0]["message"]["content"]


print("Функция ask_model готова.")

In [ ]:
answer = ask_model("Объясни, что такое промпт, одним предложением.")
print(answer)

In [ ]:
try:
    display(Image(filename="../images/notebook_after_10_minutes.png", height=400, width=500))
except Exception:
    print("Мем не нашёлся :(")

## 7. Главное: меняем промпт — меняется ответ

**Промпт** — это текст запроса к модели. Самое интересное: если поменять промпт, ответ тоже поменяется.

Ниже мы задаём модели один и тот же вопрос («что такое API»), но **просим объяснить по-разному**. Запустите и сравните ответы.


In [ ]:
prompts = [
    "Объясни, что такое API, простыми словами для ученика 5 класса.",
    "Объясни, что такое API, на примере доставки пиццы.",
    "Объясни, что такое API, как короткую шпаргалку из трёх пунктов.",
    "Объясни, что такое API, в стиле очень умного профессора со сложными словами.",
]

for prompt in prompts:
    print("=" * 70)
    print("ПРОМПТ:", prompt)
    print()
    print(ask_model(prompt))
    print()

### Мини-задание 1

Поменяйте текст в списке `prompts` ниже. Попросите модель, например:

- объяснить термин в стиле учителя информатики;
- ответить только тремя пунктами;
- привести пример из игры или из школы.

Меняйте **только текст в кавычках**, остальное не трогайте. Потом запустите ячейку.


In [ ]:
prompts = [
    "Объясни, что такое API, на примере приложения доставки еды.",
    # TODO: добавьте сюда свои варианты в кавычках, через запятую
]

for prompt in prompts:
    print("=" * 70)
    print("ПРОМПТ:", prompt)
    print()
    print(ask_model(prompt))
    print()

## 8. Просим модель отвечать в нужном формате

Модель можно просить отвечать не как угодно, а **в заданном виде**: списком, по пунктам, таблицей.

Это тоже часть промпта. Запустите и посмотрите, как модель послушалась формата.


In [ ]:
prompt = """
Объясни, что такое API.

Ответь строго по такому плану:
1. Короткое определение.
2. Пример из жизни.
3. Почему это полезно.
"""

print(ask_model(prompt))

### Мини-задание 2

Поменяйте формат ответа в промпте ниже. Например, попросите модель ответить:

- в виде таблицы;
- как пост для школьного чата;
- как конспект для подготовки к контрольной.

Меняйте только текст между `"""`.


In [ ]:
prompt = """
Объясни, что такое промпт.

Формат ответа: TODO — напишите здесь, как модель должна ответить.
"""

print(ask_model(prompt))

## 9. Маленький чат

А теперь соберём простой чат с моделью. Он работает так:

1. вы пишете сообщение;
2. сообщение добавляется в историю разговора;
3. вся история отправляется модели;
4. ответ модели печатается и тоже сохраняется в историю.

Чтобы модель «помнила» разговор, мы каждый раз отправляем ей **всю историю**.

Запустите ячейку и поговорите с моделью. Чтобы выйти, напишите `стоп`.


In [ ]:
history = ""

while True:
    user_text = input("Вы: ")
    print("Вы:", user_text)

    if user_text.lower() in ["стоп", "stop", "выход"]:
        print("Чат завершён.")
        break

    prompt = f"""Ты дружелюбный помощник на уроке.

История разговора:
{history}

Новое сообщение пользователя: {user_text}

Ответь коротко и понятно."""

    answer = ask_model(prompt)

    history += f"Пользователь: {user_text}\n"
    history += f"Помощник: {answer}\n"

    print("Модель:", answer)
    print()

## 10. Модель внутри программы: классификатор

До сих пор модель просто разговаривала с нами. Но её можно использовать и по-другому — как **помощника внутри программы**, который решает маленькую задачу.

Например: «это хорошее сообщение или спам?»

Чтобы программе было удобно, попросим модель отвечать **строго одним словом**: `GOOD` или `BAD`. Тогда дальше мы сможем написать обычный `if`.


In [ ]:
def classify_message(message):
    prompt = f"""Ты проверяешь сообщения.

Ответь СТРОГО одним словом:
GOOD — если сообщение нормальное;
BAD — если это спам, обман или грубость.

Не пиши ничего, кроме одного слова.

Сообщение: {message}"""

    answer = ask_model(prompt, temperature=0.0)
    if answer is None:
        return None
    return answer.strip().upper()


messages_to_check = [
    "Привет! Очень понравился ваш проект, хочу узнать больше.",
    "Срочно переходи по ссылке и забери приз! Только сегодня!",
    "Спасибо за объяснение, стало понятнее.",
    "Ты выиграл миллион, отправь номер карты.",
]

for message in messages_to_check:
    result = classify_message(message)
    print("Сообщение:", message)
    print("Оценка модели:", result)
    print("-" * 70)

Теперь оценку модели можно использовать в обычном `if` — как любое значение в программе.


In [ ]:
message = "Срочно переходи по ссылке и забери приз!"
result = classify_message(message)

if result == "GOOD":
    print("Публикуем сообщение")
else:
    print("Отправляем сообщение на проверку")

### А это уже агент?

Пока — **нет**. Мы сами заранее написали всю логику: «если GOOD — публикуем, иначе — на проверку». Модель только ответила на один вопрос, но **не решала, что делать дальше**.

> Не каждый вызов модели — это агент.

Агент появляется тогда, когда **модель сама выбирает следующее действие**. К этому мы придём на следующих занятиях.


In [ ]:
try:
    display(Image(filename="../images/llm_not_agent.png", width=400))
except Exception:
    print("Мем не нашёлся :(")

## 11. Что мы сегодня сделали

1. вспомнили, что модель собирает текст из кусочков-**токенов**;
2. получили ключ и временный токен;
3. отправили первый запрос и достали из ответа текст;
4. сделали удобную функцию `ask_model`;
5. **меняли промпт и видели, как меняется ответ** — это самое главное;
6. просили модель отвечать в нужном формате;
7. собрали маленький чат;
8. использовали модель как классификатор внутри программы.

Главная мысль:

> Промпт — это и есть наш способ управлять моделью. Маленькое изменение в тексте запроса сильно меняет ответ.


## 12. Домашнее задание (по желанию)

Выберите одно:

1. **Свои промпты.** Придумайте 3-4 своих варианта в списке `prompts` и сравните ответы.
2. **Свой формат.** Заставьте модель отвечать в необычном виде (стихотворение, таблица, диалог).
3. **Свой классификатор.** Поменяйте слова `GOOD`/`BAD` на свои категории, например `ВОПРОС` / `СПАМ` / `ОБЫЧНОЕ`, и проверьте список сообщений.

Главное — **запускать и смотреть**, что получается.
